In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever

# Chunk parameters
chunk_size = 256
chunk_overlap = 32

docs = CSVLoader(
    file_path='../../data/raw/hyperbook.csv', 
    content_columns=['content'], 
    metadata_columns=['url', 'title', 'authors'],
    encoding='utf-8'
).load()

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
docs = text_splitter.split_documents(docs)

sparse_retriever = BM25Retriever.from_documents(docs)

In [10]:
docs = sparse_retriever.invoke('tardigrades')
docs[0].metadata

{'source': '../../data/raw/hyperbook.csv',
 'row': 22,
 'url': 'https://bioengineering.hyperbook.mcgill.ca/the-physical-adaptations-underlying-the-tardigrades-resistance-to-extreme-environmental-conditions/',
 'title': 'The Physical Adaptations Underlying the Tardigrade’s Resistance to Extreme Environmental Conditions\xa0',
 'authors': 'Luccia Jabbour, Eden Karp-Foster, Jeremy Lachance, Wadi Zahka'}

In [ ]:
SOURCE_FORMAT = '{title}\n{authors}\n{url}'

def get_sources(docs) -> str:
    """
    Returns a string with all the sources
    formatted by title, authors and url.
    """
    sources = set() # Keep unique sources

    for doc in docs:
        metadata = doc.metadata
        source = SOURCE_FORMAT.format(
            title=metadata['title'],
            authors=metadata['authors'],
            url=metadata['url']
        )
        sources.add(source)

    # Concatenate into a string
    sources = '\n\n'.join(sources)

    return sources

In [63]:
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool

@tool('retrieval_tool', parse_docstring=True)
def multiplication_tool(int_a, int_b):
    """
    Calculates the multiplication of int_a and int_b
    
    args:
        int_a: first integer
        int_b: second integer
    """

    return int_a*int_b

model = AzureChatOpenAI(azure_deployment='gpt-4o', api_version='2024-10-21')
response = model.bind_tools([multiplication_tool]).invoke(['What is 4 times 4? Use a tool'])

In [64]:
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_gslcxmTEe0wRBJ6ER32A4F9W', 'function': {'arguments': '{"int_a":4,"int_b":4}', 'name': 'retrieval_tool'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 78, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-BrmKTfPJJe85kN7BgYVipzkaJfsIj', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}]

In [65]:
response.tool_calls

[{'name': 'retrieval_tool',
  'args': {'int_a': 4, 'int_b': 4},
  'id': 'call_gslcxmTEe0wRBJ6ER32A4F9W',
  'type': 'tool_call'}]

In [70]:
import os
import tempfile
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Frame,
    PageTemplate,
    BaseDocTemplate,
    FrameBreak,
    PageBreak,
    KeepInFrame
)

In [ ]:
title = 'Chemical Ant Adaptations'

concepts = [
    "Pheromones",
    "Exoskeletons",
    "Venom",
    "Metapleural Glands",
    "Mutualistic Fungi Relationships",
    "Soil Modification"
]

definitions = {
    "Exoskeletons": "Rigid outer structures composed of chitin and minerals, providing protection and flexibility.",
    "Metapleural Glands": "Antimicrobial secretory organs that protect colonies from pathogens.",
    "Mutualistic Fungi Relationships": "Symbiotic partnerships where ants cultivate fungi for food in controlled environments.",
    "Pheromones": "Volatile chemical signals used for communication, trail marking, and colony coordination.",
    "Soil Modification": "Alteration of soil structure through nesting activities, affecting nutrient cycling and ecosystem health.",
    "Venom": "Species-specific chemical secretions used for defense, prey immobilization, and competition."
}
equations = {}

examples = {
    "Exoskeletons": [
      "Fire ant exoskeletons resisting crushing by predators"
    ],
    "Metapleural Glands": [
      "Leaf-cutting ants applying antimicrobial secretions to fungal gardens"
    ],
    "Mutualistic Fungi Relationships": [
      "Atta ants farming Leucoagaricus fungi for sustenance"
    ],
    "Pheromones": [
      "Army ants using trail pheromones to guide foraging colonies"
    ],
    "Soil Modification": [
      "African driver ants aerating soil through massive nesting excavations"
    ],
    "Venom": [
      "Bullet ants injecting neurotoxic venom during defense"
    ]
}

In [129]:
# Create a temporary directory
temp_dir = tempfile.gettempdir()
filename = os.path.join(temp_dir, 'cheatsheet.pdf')

In [130]:
# PDF Setup
doc = BaseDocTemplate(
    filename,
    pagesize=letter,
    leftMargin=0.5*inch,
    rightMargin=0.5*inch,
    topMargin=0.5*inch,
    bottomMargin=0.5*inch
)

# Custom Styles
styles = getSampleStyleSheet()

styles.add(ParagraphStyle(
    name='CheatsheetTitle',
    parent=styles['Heading1'],
    fontSize=24,
))

styles.add(ParagraphStyle(
    name='Concept',
    parent=styles['Heading2'],
    fontSize=14,
    textColor=colors.darkblue,
    spaceAfter=0.1*inch,
    spaceBefore=0.3*inch
))

styles.add(ParagraphStyle(
    name='Subheader',
    fontSize=10,
    textColor=colors.darkred,
    spaceAfter=0.05*inch
))

styles.add(ParagraphStyle(
    name='Body',
    parent=styles['BodyText'],
    fontSize=10,
    spaceAfter=0.1*inch
))

In [131]:
# Create story content
story = []

# Add title
story.append(Paragraph(title, styles['CheatsheetTitle']))

# Calculate column dimensions
gutter = 0.2*inch
col_width = (letter[0] - doc.leftMargin - doc.rightMargin - gutter) / 2
col_height = doc.height

print(doc.width, col_width, col_height, gutter, doc.bottomMargin)

# Create frames for columns
left_frame = Frame(
    doc.leftMargin,
    doc.bottomMargin,
    col_width,
    col_height,
    id='left'
)
right_frame = Frame(
    doc.leftMargin + col_width + gutter,
    doc.bottomMargin,
    col_width,
    col_height,
    id='right'
)

# Create page template with two columns
two_column_template = PageTemplate(id='TwoColumns', frames=[left_frame, right_frame])
doc.addPageTemplates([two_column_template])

# Add content for each concept
for concept in concepts:
    # Concept header
    story.append(Paragraph(concept, styles['Concept']))
    
    # Definition
    if concept in definitions:
        story.append(Paragraph("<b>Definition</b>", styles['Subheader']))
        story.append(Paragraph(definitions[concept], styles['Body']))
    
    # Equations
    if concept in equations and equations[concept]:
        story.append(Paragraph("<b>Equations</b>", styles['Subheader']))
        for eq in equations[concept]:
            story.append(Paragraph(f"• {eq}", styles['Bullet']))
    
    # Examples
    if concept in examples and examples[concept]:
        story.append(Paragraph("<b>Examples</b>", styles['Subheader']))
        for ex in examples[concept]:
            story.append(Paragraph(f"• {ex}", styles['Bullet']))
    
    # Add spacer between concepts
    # story.append(Spacer(1, 0.2*inch))
    
    # Add frame break after every concept to maintain column balance
    # story.append(FrameBreak())

# Build the PDF document
doc.build(story)

# Return file URL (in real usage, replace with actual download URL)
print(f"file://{filename}")

540.0 262.8 720.0 14.4 36.0
file://C:\Users\dante\AppData\Local\Temp\cheatsheet.pdf
